<a href="https://colab.research.google.com/github/DaisyXUUUUU/CIS-5450-Project/blob/main/notebooks/01_data_part_presentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 1 — Data: Acquisition, Sampling, Cleaning, Joins, and Split

**Owner:** Ziyue Xu (Daisy) · CIS 5450 Final Project · *Tipping Behavior in NYC Yellow Taxis: Patterns and Prediction Among Trips with Recorded Card Tips*

This notebook **rebuilds the analysis table from scratch**: it downloads the raw data from the official links, draws the
sample with DuckDB, cleans it, joins zones and weather, adds the features, and splits by time. Every section explains the
data decision it makes (problem → decision → reason) and shows the result.

Safety net:

- The raw files are downloaded to the **Colab disk** (`/content/fresh/`). Nothing is written to Google Drive.
- At the end, the rebuilt table is compared, row by row and column by column, with the fixed table the rest of the
  project uses (`trips_clean_v2.parquet` on Drive, read only). It must be the same **796,989 trips**.
- If a download fails (e.g. no internet during the demo), the notebook falls back to the copies in `MyDrive/.../data/raw/`
  (copied to the Colab disk first, because DuckDB on the Drive mount is very slow).

| Section | Topic |
| --- | --- |
| 0 | Research question, target, leakage rule, pipeline, setup |
| 1 | Data acquisition |
| 2 | Sampling with DuckDB (and the reservoir-sampling pitfall) |
| 3 | Data quality check before cleaning |
| 4 | Cleaning rules and row funnel |
| 5 | Joins: taxi zones and hourly weather |
| 6 | Features: proposal Table 1, target, and v2 columns |
| 7 | Money-field check (`extra`, bill) |
| 8 | Time-based train / validation / test split |
| 9 | Output, data dictionary, check against the fixed table, summary |

Correlation checks and EDA (Yuzhi), class balance and leakage analysis for modeling (Yiwen), and encoding / scaling (modeling pipeline) are in teammates' notebooks.

## 0. Research question, target, and the leakage rule

**Question.** Among NYC yellow taxi trips with a positive recorded card tip, which time, location, trip, and weather
characteristics are associated with a high tip, and how well can a high tip be predicted?

**Target.** `high_tip = 1` if `tip_amount / fare_amount ≥ 25%` (proposal; 20% and 30% kept for sensitivity).
Section 6 adds the bill-based alternative `high_tip_bill` (tip as a share of everything paid before the tip).

**Model time point: the trip has ended, the rider has not paid yet.** So anything known at drop-off (time, place,
distance, duration, fare, surcharges, weather) may be a feature, but anything decided at payment may not:

> `tip_amount` and `total_amount` (which includes the tip), and every column computed from them (`tip_pct`, `high_tip*`,
> `tip_pct_bill`, `high_tip_bill*`), are **never** model features. They exist only to build the target.

**Pipeline.**

```
TLC monthly Parquet (Apr–Jun 2026, 11.76M trips)
  └─ DuckDB: card trips only, 300,000 per month, seed 5450  ──►  900,000 sampled trips
       └─ 8 cleaning rules, logged one by one                  ──►  796,989 trips
            └─ LEFT JOIN taxi zones (pickup + drop-off), LEFT JOIN hourly weather
                 └─ features, target, v2 columns, time split    ──►  796,989 × 54 analysis table
```

**Run record.**

| Item | Value |
| --- | --- |
| Original download and run | 2026-11-03 (`01_data_pipeline.ipynb`, commit `e28d7d4`); v2 columns 2026-11-16 (`01b_add_features_v2.ipynb`) |
| Reproducibility test | 2026-11-17, `01c_reproducibility_fresh_download.ipynb`: a fresh download gave the identical table (all checks passed) |
| Random seed | `SEED = 5450` (the only random step is the sample) |
| Software | DuckDB 1.3.2 (the sample uses DuckDB's `hash()`, which can change between versions), pandas 2.2.3, Google Colab |

In [ ]:
import os, sys, time, shutil, subprocess, tempfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")      # only read: the fixed table to compare with, and backup raw copies

try:
    import duckdb
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb==1.3.2"], check=True)
    import duckdb
import numpy as np
import pandas as pd
import requests
from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 200)
print("duckdb", duckdb.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)
if duckdb.__version__ != "1.3.2":
    print("WARNING: the sample was designed with DuckDB 1.3.2; hash() may select different trips in this version.")

### Parameters

All design choices live in this one cell.

In [ ]:
# ---- study design (fixed in the proposal) --------------------------------------------
SEED = 5450
MONTHS = ["2026-04", "2026-05", "2026-06"]
N_PER_MONTH = 300_000
STUDY_START, STUDY_END = pd.Timestamp("2026-04-01"), pd.Timestamp("2026-07-01")   # [start, end)
TRAIN_END = pd.Timestamp("2026-05-16")   # train: Apr 1 – May 15
VAL_END   = pd.Timestamp("2026-06-01")   # val:   May 16 – May 31; test: June
HIGH_TIP_THRESHOLD, SENSITIVITY_THRESHOLDS = 0.25, [0.20, 0.30]
RAIN_MM = 0.1
NYC_LAT, NYC_LON = 40.7128, -74.0060
JFK, LGA, EWR = 132, 138, 1
UNKNOWN_ZONES = [264, 265]

DEMO_RESERVOIR_PITFALL = False   # True = also run the first (rejected) reservoir sample on May to show its time bias

# ---- where things go -----------------------------------------------------------------
FRESH_DIR = Path("/content/fresh") if IN_COLAB else Path(tempfile.gettempdir()) / "cis5450_fresh"
FRESH_DIR.mkdir(parents=True, exist_ok=True)
RAW = {m: FRESH_DIR / f"yellow_tripdata_{m}.parquet" for m in MONTHS}
ZONES_FILE, WEATHER_FILE = FRESH_DIR / "taxi_zone_lookup.csv", FRESH_DIR / "weather_hourly_nyc.csv"

DATA_DIR = Path(os.environ.get("CIS5450_DATA_DIR", "/content/drive/MyDrive/CIS5450_project/data"))   # read only
DRIVE_RAW = DATA_DIR / "raw"
FIXED_V2 = DATA_DIR / "clean" / "trips_clean_v2.parquet"

# ---- results of the original run that this rebuild must reproduce ----------------------
EXPECTED_ROWS = 796_989
EXPECTED_SPLIT_ROWS = {"train": 395_144, "val": 127_349, "test": 274_496}
EXPECTED_SPLIT_HIGH_TIP = {"train": 0.633, "val": 0.624, "test": 0.630}
print("raw files ->", FRESH_DIR, "| fixed table to compare with:", FIXED_V2, "exists:", FIXED_V2.exists())

## 1. Data acquisition

**Sources.**

| Data | Source | Grain | Used for |
| --- | --- | --- | --- |
| Yellow taxi trip records, April–June 2026 | [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page), one Parquet file per month | one row per trip | everything |
| Taxi Zone Lookup Table | same page (`taxi_zone_lookup.csv`) | one row per zone (265) | borough / zone names for pickup and drop-off |
| Hourly weather, Manhattan | [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api) (`temperature_2m`, `precipitation`, New York time) | one row per hour (2,184) | temperature, rain flag |

**Why these months.** April–June 2026 are three recent consecutive months, all after the CBD congestion fee began
(2025-01-05), so the fare and surcharge rules are the same across train, validation, and test. Three months give ~11.8
million trips, far above the course minimum.

In [ ]:
TLC_BASE = os.environ.get("CIS5450_TLC_BASE", "https://d37ci6vzurychx.cloudfront.net")
SOURCES = {f"yellow_tripdata_{m}.parquet": f"{TLC_BASE}/trip-data/yellow_tripdata_{m}.parquet" for m in MONTHS}
SOURCES["taxi_zone_lookup.csv"] = f"{TLC_BASE}/misc/taxi_zone_lookup.csv"

def download(url, dest):
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        tmp = dest.with_suffix(dest.suffix + ".part")
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
        tmp.rename(dest)

def download_weather(dest):
    resp = requests.get("https://archive-api.open-meteo.com/v1/archive", timeout=120, params={
        "latitude": NYC_LAT, "longitude": NYC_LON, "start_date": "2026-04-01", "end_date": "2026-06-30",
        "hourly": "temperature_2m,precipitation", "timezone": "America/New_York"})
    resp.raise_for_status()
    pd.DataFrame(resp.json()["hourly"]).to_csv(dest, index=False)

ACQUIRED_FROM = {}
jobs = [(name, (lambda u=url, n=name: download(u, FRESH_DIR / n))) for name, url in SOURCES.items()]
jobs.append(("weather_hourly_nyc.csv", lambda: download_weather(WEATHER_FILE)))
for name, job in jobs:
    dest = FRESH_DIR / name
    if dest.exists() and dest.stat().st_size > 0:
        ACQUIRED_FROM[name] = "already downloaded in this session"
    else:
        t0 = time.time()
        try:
            job()
            ACQUIRED_FROM[name] = f"official link ({time.time() - t0:.0f} s)"
        except Exception as e:                       # no internet / site down: use the Drive copy (read only)
            backup = DRIVE_RAW / name
            if not backup.exists():
                raise RuntimeError(f"Download of {name} failed and there is no backup at {backup}") from e
            shutil.copyfile(backup, dest)
            ACQUIRED_FROM[name] = f"DOWNLOAD FAILED ({type(e).__name__}); copied from Drive backup"
    print(f"{name:34s} {dest.stat().st_size / 1e6:7.1f} MB   <- {ACQUIRED_FROM[name]}")
print("Acquired (UTC):", pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d %H:%M"))

**Size of the raw data.** Row and column counts come from the Parquet file footer; the card-trip count is a DuckDB query
on one column. Nothing is loaded into pandas.

In [ ]:
import pyarrow.parquet as pq
con = duckdb.connect()

RECORDED_RAW = {"rows": [3_831_240, 4_090_836, 3_837_248], "card_trips": [2_635_749, 2_727_585, 2_432_871]}
rows = []
for m in MONTHS:
    meta = pq.ParquetFile(RAW[m]).metadata
    n_card = con.execute(f"SELECT count(*) FROM read_parquet('{RAW[m]}') WHERE payment_type = 1").fetchone()[0]
    rows.append({"month": m, "rows": meta.num_rows, "columns": meta.num_columns, "card_trips": n_card})
raw_summary = pd.DataFrame(rows)
raw_summary["card_share"] = raw_summary["card_trips"] / raw_summary["rows"]
display(raw_summary.style.format({"rows": "{:,}", "card_trips": "{:,}", "card_share": "{:.1%}"}))
print(f"Total raw trips: {raw_summary['rows'].sum():,} | card trips: {raw_summary['card_trips'].sum():,}")
print("Same counts as the original 2026-11-03 download:",
      raw_summary["rows"].tolist() == RECORDED_RAW["rows"] and raw_summary["card_trips"].tolist() == RECORDED_RAW["card_trips"])

In [ ]:
# Raw schema (June). June has one extra column, `request_source`, that is not in the TLC data dictionary.
con.execute(f"DESCRIBE SELECT * FROM read_parquet('{RAW['2026-06']}')").df()[["column_name", "column_type"]]

**Conclusion.** 11,759,324 raw trips; 63–69% are credit-card trips. We only use card trips, because TLC records tips
only for card payments (a cash tip is always stored as 0).

## 2. Sampling with DuckDB

**Problem.** 7.8 million card trips is more than we need and slow to iterate on in Colab (and some models, e.g. the
FT-Transformer, are expensive). We want a smaller sample that is still representative of every day and hour.

**Decision.** From each month keep exactly **300,000 card trips** (`payment_type = 1`), drawn uniformly at random with
`SEED = 5450` → 900,000 trips. DuckDB queries the Parquet files directly, so the raw rows never enter pandas.

How: each card trip gets a pseudo-random key `hash(file_row_number, SEED)`; we keep the 300,000 trips with the smallest keys.
The key does not depend on the trip's content or its position in the file, so every card trip has the same chance, and the
same seed returns the same trips.

**Pitfall we hit first (and why the check below exists).** The first version used
`SELECT * FROM (... WHERE payment_type = 1) USING SAMPLE reservoir(300000 ROWS) REPEATABLE (5450)`.
TLC files are roughly sorted by time, and that sample came out concentrated in the first half of each month: after cleaning,
the May 16–31 **validation fold had only 10,836 trips** (it should have ~127,000). Nothing crashed; we only noticed from the
fold sizes. That run was discarded. We switched to hash sampling and added an automatic check that compares the share of
trips on each day in the sample with the full month.

**Equal 300k per month** (rather than proportional): months differ by under 7% in card trips, and equal counts keep the
months balanced. Day-of-month shares *within* a month stay proportional, which is what the check verifies.

In [ ]:
SAMPLE_SQL = '''
    SELECT * EXCLUDE (file_row_number)
    FROM read_parquet('{path}', file_row_number = true)
    WHERE payment_type = 1                         -- card trips only
    ORDER BY hash(file_row_number, {seed})         -- seeded pseudo-random key
    LIMIT {n}
'''
t0 = time.time()
parts = []
for m in MONTHS:
    part = con.execute(SAMPLE_SQL.format(path=RAW[m], seed=SEED, n=N_PER_MONTH)).df()
    part["source_month"] = m
    parts.append(part)
sample = pd.concat(parts, ignore_index=True)
for c in ["tpep_pickup_datetime", "tpep_dropoff_datetime"]:
    sample[c] = pd.to_datetime(sample[c])
print(f"Sample: {len(sample):,} rows x {sample.shape[1]} columns in {time.time() - t0:.0f} s")
print(sample["source_month"].value_counts().sort_index().to_string())
assert sample["source_month"].value_counts().eq(N_PER_MONTH).all() and len(sample) == 3 * N_PER_MONTH

### Check: is the sample spread evenly over each month?

For each month, compare the share of card trips on each day in the full file with the share in the sample.
With 300,000 trips per month an unbiased sample stays within ~0.1–0.15 percentage points on every day; we stop if any
day is off by more than 0.3 points (the reservoir sample was off by several points).

In [ ]:
files = ", ".join(repr(str(RAW[m])) for m in MONTHS)
pop_daily = con.execute(f'''
    SELECT CAST(tpep_pickup_datetime AS DATE) AS day, count(*) AS n
    FROM read_parquet([{files}], union_by_name = true)
    WHERE payment_type = 1
    GROUP BY ALL
''').df()
pop_daily["day"] = pd.to_datetime(pop_daily["day"])
samp_daily = (sample.assign(day=sample["tpep_pickup_datetime"].dt.normalize())
                    .groupby(["source_month", "day"]).size().rename("n_sample").reset_index())
rows = []
for m in MONTHS:
    p = pop_daily[pop_daily["day"].dt.strftime("%Y-%m") == m].set_index("day")["n"]
    q = samp_daily[(samp_daily["source_month"] == m) & (samp_daily["day"].dt.strftime("%Y-%m") == m)].set_index("day")["n_sample"]
    share = pd.concat([p / p.sum(), q / q.sum()], axis=1, keys=["full_month", "sample"]).fillna(0)
    first_half = share.index.day <= 15
    rows.append({"month": m,
                 "full_month_share_days_1_15": share.loc[first_half, "full_month"].sum(),
                 "sample_share_days_1_15": share.loc[first_half, "sample"].sum(),
                 "max_daily_gap_pct_points": (share["full_month"] - share["sample"]).abs().max() * 100})
daily_check = pd.DataFrame(rows)
display(daily_check.round(4))
assert (daily_check["max_daily_gap_pct_points"] < 0.3).all(), "Sample is not spread evenly over the month"
print("Every day within 0.3 percentage points: True")

In [ ]:
# Optional demo of the rejected method (off by default; it is not used anywhere).
if DEMO_RESERVOIR_PITFALL:
    con.execute("SET threads = 1")
    bad = con.execute(f'''
        SELECT tpep_pickup_datetime FROM (
            SELECT * FROM read_parquet('{RAW["2026-05"]}') WHERE payment_type = 1
        ) USING SAMPLE reservoir({N_PER_MONTH} ROWS) REPEATABLE ({SEED})
    ''').df()
    con.execute("RESET threads")
    late = (pd.to_datetime(bad["tpep_pickup_datetime"]).dt.day >= 16).mean()
    print(f"Reservoir sample, May: share of trips on May 16-31 = {late:.1%} (full month: about 49%)")
else:
    print("Demo off. Recorded result of the rejected reservoir sample: validation fold (May 16-31) = 10,836 trips after cleaning.")

**Conclusion.** 900,000 card trips, exactly 300,000 per month, and each day's share in the sample is within about 0.06
percentage points of the full month. This is the only random step in the project.

## 3. Data quality check before cleaning

**Goal.** Look at the 900,000 sampled trips *before* any rule is applied: column types, missing values, value ranges, and
how many trips show each kind of problem. Each problem below is counted on the whole sample independently (one trip can
have several problems), so these counts overlap; Section 4 removes them in order without double counting.

In [ ]:
pd.DataFrame({"dtype": sample.dtypes.astype(str),
              "missing": sample.isna().sum(),
              "missing_pct": (sample.isna().mean() * 100).round(2)})

In [ ]:
q = sample
q_duration = (q["tpep_dropoff_datetime"] - q["tpep_pickup_datetime"]).dt.total_seconds() / 60
num = q[["trip_distance", "fare_amount", "tip_amount", "total_amount", "extra", "tolls_amount", "passenger_count"]].assign(duration_min=q_duration)
num.describe(percentiles=[0.01, 0.5, 0.99]).T.round(2)

In [ ]:
with np.errstate(divide="ignore", invalid="ignore"):
    q_tip_ratio = q["tip_amount"] / q["fare_amount"].where(q["fare_amount"] > 0)
issues = pd.Series({
    "tip_amount = 0 (no card tip recorded)":    (q["tip_amount"] == 0).sum(),
    "tip_amount < 0":                            (q["tip_amount"] < 0).sum(),
    "fare_amount <= 0":                          (q["fare_amount"] <= 0).sum(),
    "pickup outside Apr 1 - Jun 30, 2026":       (~q["tpep_pickup_datetime"].between(STUDY_START, STUDY_END, inclusive="left")).sum(),
    "duration < 1 min":                          (q_duration < 1).sum(),
    "duration > 180 min":                        (q_duration > 180).sum(),
    "trip_distance = 0":                         (q["trip_distance"] <= 0).sum(),
    "trip_distance > 100 mi":                    (q["trip_distance"] > 100).sum(),
    "pickup or drop-off zone 264/265 (unknown)": (q["PULocationID"].isin(UNKNOWN_ZONES) | q["DOLocationID"].isin(UNKNOWN_ZONES)).sum(),
    "RatecodeID = 99 (unknown)":                 (q["RatecodeID"] == 99).sum(),
    "tip > fare (tip/fare > 100%)":              (q_tip_ratio > 1).sum(),
    "passenger_count = 0":                       (q["passenger_count"] == 0).sum(),
    "request_source not empty":                  q["request_source"].notna().sum() if "request_source" in q else 0,
}, name="trips").to_frame()
issues["pct_of_sample"] = (issues["trips"] / len(q) * 100).round(2)
issues

**What we found and what we do about it.**

| Issue | Decision | Reason |
| --- | --- | --- |
| `tip_amount = 0` (~8%) | drop | Study population is trips **with a recorded card tip**. A zero card tip cannot tell "did not tip" from "tipped in cash", so it is not a reliable "no tip" label. |
| `fare_amount ≤ 0` (incl. negative refunds) | drop | Refunds / voids / errors; tip ÷ fare is undefined. |
| pickup outside April–June | drop | TLC files contain a few trips with wrong timestamps. |
| duration < 1 min or > 180 min | drop | Under a minute is a meter start/stop error; over three hours is not a single street-hail trip. |
| distance = 0 or > 100 mi | drop | Zero is a meter / GPS error; > 100 mi is outside NYC yellow-cab service. |
| zone 264 / 265 | drop | "Unknown" / "Outside of NYC": cannot be mapped to a borough, so location features would be meaningless. |
| `RatecodeID = 99` | drop | Unknown rate code. All such trips are already removed by earlier rules (breakdown in Section 4). |
| tip > fare | drop | Mostly keying errors (e.g. tip typed in the fare field); a handful of extreme ratios would dominate every ratio summary. |
| `passenger_count = 0` | **keep the trip**, set count to missing | Only the count is wrong, the trip is fine. Imputation happens in the modeling pipeline on train only. |
| `request_source` | drop the **column** | Not in the TLC data dictionary, absent in April–May, and empty for every sampled card trip in June. |
| very slow trips (short distance, long time) | **keep** | Could be real traffic; see Section 4. |

All thresholds are fixed constants from domain knowledge, not quantiles of the data, so June (test) data does not influence any rule.

## 4. Cleaning: rules and row funnel

Rules are applied one at a time in this order, and each step logs how many trips it removed (same code as `01_data_pipeline.ipynb`).

In [ ]:
RULES = [
    ("1. tip_amount > 0",                     lambda d: d["tip_amount"] > 0),
    ("2. fare_amount > 0",                    lambda d: d["fare_amount"] > 0),
    ("3. pickup within Apr 1 - Jun 30, 2026", lambda d: d["tpep_pickup_datetime"].between(STUDY_START, STUDY_END, inclusive="left")),
    ("4. 1 <= duration <= 180 min",           lambda d: d["duration_min"].between(1, 180)),
    ("5. 0 < trip_distance <= 100 mi",        lambda d: (d["trip_distance"] > 0) & (d["trip_distance"] <= 100)),
    ("6. PU/DO zone not 264/265",             lambda d: ~d["PULocationID"].isin(UNKNOWN_ZONES) & ~d["DOLocationID"].isin(UNKNOWN_ZONES)),
    ("7. RatecodeID != 99",                   lambda d: d["RatecodeID"].ne(99) | d["RatecodeID"].isna()),
    ("8. tip_pct <= 100%",                    lambda d: d["tip_amount"] / d["fare_amount"] <= 1.0),
]

df = sample.drop(columns=["request_source"], errors="ignore").copy()
df["duration_min"] = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60
log = [{"step": "0. sampled card trips", "rows_after": len(df), "removed": 0}]
for name, rule in RULES:
    before = len(df)
    df = df[rule(df).fillna(False).astype(bool)]
    log.append({"step": name, "rows_after": len(df), "removed": before - len(df)})
df = df.reset_index(drop=True)
funnel = pd.DataFrame(log)
funnel["pct_removed_of_sample"] = funnel["removed"] / funnel.loc[0, "rows_after"] * 100
funnel["pct_kept_so_far"] = funnel["rows_after"] / funnel.loc[0, "rows_after"] * 100
display(funnel.style.format({"rows_after": "{:,}", "removed": "{:,}", "pct_removed_of_sample": "{:.2f}", "pct_kept_so_far": "{:.2f}"}))

RECORDED_FUNNEL = [900_000, 825_235, 825_233, 825_232, 804_723, 802_836, 798_194, 798_194, 796_989]
print("Funnel identical to the original 2026-11-03 run:", funnel["rows_after"].tolist() == RECORDED_FUNNEL)
assert len(df) == EXPECTED_ROWS, f"expected {EXPECTED_ROWS:,} rows after cleaning, got {len(df):,}"

In [ ]:
# Rule 7 removes 0 trips although some sampled trips have RatecodeID = 99. Which earlier rule removed them?
r99 = sample[sample["RatecodeID"] == 99].copy()
r99["duration_min"] = (r99["tpep_dropoff_datetime"] - r99["tpep_pickup_datetime"]).dt.total_seconds() / 60
first_failed = pd.Series("passes rules 1-6", index=r99.index)
for name, rule in reversed(RULES[:6]):            # reversed, so the earliest failing rule wins
    first_failed[~rule(r99).fillna(False).astype(bool)] = name
print(f"RatecodeID = 99 in the sample: {len(r99):,} trips. First rule that removes them:")
print(first_failed.value_counts().to_string())

Reading the funnel: after rules 1–2 (positive tip and fare) **825,233** trips remain (the proposal reported 825,072 from
the first, discarded sample; the small gap is from re-drawing with the unbiased method). Duration is the only other rule
that removes more than 1%. Final: **796,989 trips (88.6% of the sample)**.

### Things we deliberately did *not* remove

**Slow trips (short distance, long duration).** A 1-mile trip that takes 40 minutes looks odd, but in Manhattan rush hour
it can be real traffic. We have no ground truth to tell a meter error from a traffic jam, and dropping them would remove
exactly the congested trips that matter for the rush-hour question. So only the 1–180 minute rule applies.

**`passenger_count = 0`** becomes missing; the trip stays.

In [ ]:
n_zero_pass = int((df["passenger_count"] == 0).sum())
df.loc[df["passenger_count"] == 0, "passenger_count"] = np.nan
print(f"passenger_count = 0 set to missing: {n_zero_pass:,} ({n_zero_pass / len(df):.2%}) -- imputed later on train only")

speed = 60 * df["trip_distance"] / df["duration_min"]
pu_h, pu_d = df["tpep_pickup_datetime"].dt.hour, df["tpep_pickup_datetime"].dt.dayofweek
rush = (pu_d <= 4) & pu_h.between(16, 19)
slow = speed < 3
print(f"Trips slower than 3 mph: {slow.sum():,} ({slow.mean():.2%}) -- kept")
print(f"  share in weekday 16-19h:      {rush[slow].mean():.1%}  (all trips: {rush.mean():.1%})")
print(f"  median distance / duration:   {df.loc[slow, 'trip_distance'].median():.2f} mi / {df.loc[slow, 'duration_min'].median():.1f} min")
print(f"Trips faster than 80 mph: {int((speed > 80).sum())} -- kept here; any cap is chosen in feature engineering on train only")

**Conclusion.** 796,989 clean trips, every removal explained and counted. No values were imputed, capped, or scaled here:
those steps learn from the data and therefore belong in the modeling pipeline, fitted on the training split only.

## 5. Joins: taxi zones and hourly weather

**Zone join.** `PULocationID` and `DOLocationID` are each LEFT JOINed to the zone lookup (two joins, one per trip end),
adding borough, zone name, and service zone. LEFT JOIN so that a missing match shows up as a null we can count, instead of
silently dropping the trip.

**Weather join — alignment rule (no future information).** Open-Meteo labels hour *T* with the precipitation that fell
from *T − 1h* to *T*. We floor each pickup time to the hour (`pickup_hour_ts`) and join on it, which picks the most recent
hour that had ended by pickup. Rounding to the nearest hour would sometimes use an hour that had not finished yet.
TLC timestamps are New York local time; the weather is requested in `America/New_York`; April–June 2026 has no
daylight-saving change (it was March 8), so every local hour is unique.

`is_rain = precip_mm ≥ 0.1` (smallest amount the API reports as non-zero rain).

In [ ]:
zones = pd.read_csv(ZONES_FILE)
print(f"Zone lookup: {len(zones)} zones")
con.register("trips", df)
con.register("zones", zones)
n_before = len(df)
df = con.execute('''
    SELECT t.*,
           pz.Borough AS PU_Borough, pz.Zone AS PU_Zone, pz.service_zone AS PU_service_zone,
           dz.Borough AS DO_Borough, dz.Zone AS DO_Zone, dz.service_zone AS DO_service_zone
    FROM trips AS t
    LEFT JOIN zones AS pz ON t.PULocationID = pz.LocationID
    LEFT JOIN zones AS dz ON t.DOLocationID = dz.LocationID
''').df()
assert len(df) == n_before, "zone join changed the row count"
print(f"Rows before/after zone join: {n_before:,} / {len(df):,}")
print("Trips with no pickup borough:", int(df["PU_Borough"].isna().sum()), "| no drop-off borough:", int(df["DO_Borough"].isna().sum()))
assert df["PU_Borough"].notna().all() and df["DO_Borough"].notna().all()
pd.crosstab(df["PU_Borough"], df["DO_Borough"], margins=True)

In [ ]:
weather = pd.read_csv(WEATHER_FILE, parse_dates=["time"]).rename(
    columns={"time": "weather_hour", "temperature_2m": "temp_c", "precipitation": "precip_mm"})
expected_hours = (STUDY_END - STUDY_START) // pd.Timedelta(hours=1)
print(f"Weather hours: {len(weather):,} (expected {expected_hours:,}), duplicates: {weather['weather_hour'].duplicated().sum()}, "
      f"missing values: {int(weather[['temp_c', 'precip_mm']].isna().sum().sum())}")
print(f"Rainy hours (>= {RAIN_MM} mm): {(weather['precip_mm'] >= RAIN_MM).sum()} of {len(weather)}")
assert len(weather) == expected_hours and not weather["weather_hour"].duplicated().any()

df["pickup_hour_ts"] = df["tpep_pickup_datetime"].dt.floor("h")
con.register("trips_z", df)
con.register("weather", weather)
df = con.execute('''
    SELECT t.*, w.temp_c, w.precip_mm
    FROM trips_z AS t
    LEFT JOIN weather AS w ON t.pickup_hour_ts = w.weather_hour
''').df()
assert len(df) == n_before, "weather join changed the row count"
df["is_rain"] = (df["precip_mm"] >= RAIN_MM).astype("Int8").where(df["precip_mm"].notna())

assert (df["pickup_hour_ts"] <= df["tpep_pickup_datetime"]).all()     # never a later hour than pickup
coverage = df["temp_c"].notna().mean()
print(f"Weather coverage: {coverage:.2%} | trips in a rainy hour: {int(df['is_rain'].sum()):,} ({df['is_rain'].mean():.1%})")
assert coverage == 1.0

**Conclusion.** Both joins keep the row count (796,989) and match **100%** of trips: every trip has a pickup and drop-off
borough and an hour of weather. 16.2% of trips started in a rainy hour (319 of 2,184 hours had ≥ 0.1 mm).
91% of pickups are in Manhattan and most of the rest in Queens (the airports), so outer-borough results will be less precise.

## 6. Features: proposal Table 1, target, and v2 columns

Every column here is a deterministic function of the trip's own record, its zones, or the weather hour that ended before
pickup. No statistic is learned from the data, so building them before the split cannot leak test information.

**Table 1 features and target.**

| Column | Definition |
| --- | --- |
| `pickup_hour`, `pickup_dow` | hour 0–23; day of week, 0 = Monday |
| `is_weekend` | Saturday or Sunday |
| `is_late_night` | pickup 00:00–05:59 |
| `avg_speed_mph` | 60 × `trip_distance` / `duration_min` |
| `is_airport` / `is_newark` | pickup or drop-off at JFK (132) or LaGuardia (138) / at Newark (1) |
| `same_borough`, `has_cbd_fee`, `is_short_trip` | same pickup and drop-off borough; CBD fee charged; distance ≤ 2 mi |
| `tip_pct`, `high_tip`, `high_tip_20`, `high_tip_30` | tip ÷ fare and the 25% target (20% / 30% sensitivity). **Not features.** |
| `pickup_date` | cluster unit for date-clustered SEs and the day-block bootstrap |

In [ ]:
pu = df["tpep_pickup_datetime"]
df["pickup_hour"]   = pu.dt.hour.astype("int8")
df["pickup_dow"]    = pu.dt.dayofweek.astype("int8")
df["is_weekend"]    = (df["pickup_dow"] >= 5).astype("int8")
df["is_late_night"] = (df["pickup_hour"] <= 5).astype("int8")
df["avg_speed_mph"] = 60 * df["trip_distance"] / df["duration_min"]
df["is_airport"]    = (df["PULocationID"].isin([JFK, LGA]) | df["DOLocationID"].isin([JFK, LGA])).astype("int8")
df["is_newark"]     = (df["PULocationID"].eq(EWR) | df["DOLocationID"].eq(EWR)).astype("int8")
df["same_borough"]  = (df["PU_Borough"] == df["DO_Borough"]).astype("int8")
df["has_cbd_fee"]   = (df["cbd_congestion_fee"].fillna(0) > 0).astype("int8")
df["is_short_trip"] = (df["trip_distance"] <= 2).astype("int8")

df["tip_pct"] = df["tip_amount"] / df["fare_amount"]
df["high_tip"] = (df["tip_pct"] >= HIGH_TIP_THRESHOLD).astype("int8")
for t in SENSITIVITY_THRESHOLDS:
    df[f"high_tip_{int(t * 100)}"] = (df["tip_pct"] >= t).astype("int8")
df["pickup_date"] = pu.dt.normalize()

flags = ["is_weekend", "is_late_night", "is_airport", "is_newark", "same_borough", "has_cbd_fee", "is_short_trip", "high_tip"]
df[flags].mean().rename("share of trips").to_frame().style.format("{:.1%}")

**v2 columns** (added on 2026-11-16 in `01b_add_features_v2.ipynb`, same code here).

| Column | Definition | Why |
| --- | --- | --- |
| `is_rush_hour` | Mon–Fri, pickup 16:00–19:59 | TLC's $2.50 rush-hour surcharge window (keyed on pickup time). Feature. |
| `is_night` | pickup 20:00–05:59 | TLC's $1 overnight surcharge window. Feature. |
| `tip_pct_bill` | `tip_amount / (total_amount − tip_amount)` | Tip as a share of everything the rider paid before the tip, i.e. what the payment screen's % buttons are applied to. Outcome, **not a feature**. |
| `high_tip_bill` | tip > `round(0.20 × bill, 2) + $0.015` | Tip above the 20% preset by more than a cent (rounding to the cent would otherwise push preset tips to either side of 0.20). Alternative target, **not a feature**. |
| `high_tip_bill_25`, `_30` | same rule at 25% / 30% | Sensitivity checks, **not features**. |

**Holidays are not excluded from `is_rush_hour`.** The TLC rule text says the rush surcharge does not apply on legal
holidays, but on the two federal holidays in our window (Memorial Day, Mon May 25; Juneteenth, Fri June 19) about 90% of
Vendor 2 trips between 16:00 and 19:59 were charged `extra = 2.50`, close to the ~98% of an ordinary weekday. The flag
follows what riders were actually charged (check printed below).

Why the bill-based denominator was added at all is part of the EDA (Yuzhi): the hour-of-day pattern in tip ÷ fare is
largely driven by the surcharges in the denominator.

In [ ]:
hour, dow = pu.dt.hour, pu.dt.dayofweek
bill = (df["total_amount"] - df["tip_amount"]).round(2)
v2 = {}
v2["is_rush_hour"] = ((dow <= 4) & hour.between(16, 19)).astype("int8")
v2["is_night"] = ((hour >= 20) | (hour <= 5)).astype("int8")
v2["tip_pct_bill"] = pd.Series(np.where(bill > 0, df["tip_amount"] / bill, np.nan), index=df.index)
for t, col in [(0.20, "high_tip_bill"), (0.25, "high_tip_bill_25"), (0.30, "high_tip_bill_30")]:
    s = (df["tip_amount"] > (t * bill).round(2) + 0.015).astype("int8")
    s[bill <= 0] = 0
    v2[col] = s
V2_NEW = list(v2)
# held aside and attached after the split column (Section 8), so the column order matches the saved v2 file

assert not (v2["is_rush_hour"].astype(bool) & v2["is_night"].astype(bool)).any(), "rush and night overlap"
display(pd.DataFrame(v2).describe().T[["mean", "min", "50%", "max"]].round(4))

HOLIDAYS = pd.to_datetime(["2026-05-25", "2026-06-19"])   # Memorial Day (Mon), Juneteenth (Fri)
is_v2 = df["VendorID"] == 2                               # Vendor 1's `extra` also bundles other fees (Section 7)
is_hol = pu.dt.normalize().isin(HOLIDAYS)
win = hour.between(16, 19) & (dow <= 4)
print(f"Vendor 2, weekday 16-19h, share charged extra >= 2.50:  ordinary weekdays {(df.loc[is_v2 & win & ~is_hol, 'extra'] >= 2.5).mean():.1%}"
      f"  |  the two holidays {(df.loc[is_v2 & win & is_hol, 'extra'] >= 2.5).mean():.1%}  ({int((is_v2 & win & is_hol).sum())} trips)")

**Conclusion.** `high_tip` (≥ 25% of fare) ≈ 63% of trips. `is_rush_hour` ≈ 21% and `is_night` ≈ 28%; `high_tip_bill`
≈ 16% of trips, because about half of all riders tip exactly the 20% preset.

## 7. Money-field check: `extra` and the bill

**Problem.** If the fare components are added up, they should equal `total_amount − tip_amount`. For Vendor 2 they do; for
Vendor 1 (Creative Mobile, ~18% of trips) they almost never do.

**Finding.** Vendor 1's `extra` already **includes** the congestion surcharge ($2.50), the CBD fee ($0.75), and the airport
fee, which also appear in their own columns. Summing everything therefore double-counts those fees for Vendor 1.

**Decision.** No rows or values change (no cleaning rule used `extra`). The bill is always computed as
`bill = total_amount − tip_amount`, which needs no component sum. If `extra` is used as a model feature, Vendor 1's
value must first be reduced by those three fees so both vendors mean the same thing.

In [ ]:
pre_tip = df["total_amount"] - df["tip_amount"]
base = df[["fare_amount", "extra", "mta_tax", "tolls_amount", "improvement_surcharge"]].sum(axis=1)
fees = df[["congestion_surcharge", "cbd_congestion_fee", "Airport_fee"]].fillna(0).sum(axis=1)
reconciles = lambda s: (s - pre_tip).abs() <= 0.011

recon = pd.DataFrame({
    "trips": df["VendorID"].value_counts(),
    "sum incl. 3 fee columns = bill": reconciles(base + fees).groupby(df["VendorID"]).mean(),
    "sum excl. 3 fee columns = bill": reconciles(base).groupby(df["VendorID"]).mean(),
}).sort_index()
display(recon.style.format({"trips": "{:,}", "sum incl. 3 fee columns = bill": "{:.1%}", "sum excl. 3 fee columns = bill": "{:.1%}"}))

print("Most common `extra` values, Vendor 1 (3.25 = 2.50 + 0.75; 5.75 = + 2.50 rush):")
print(df.loc[df["VendorID"] == 1, "extra"].round(2).value_counts().head(6).to_string())
print("Trips with bill < fare:", int((pre_tip.round(2) < df["fare_amount"]).sum()))

**Conclusion.** Vendor 1: almost no trips reconcile when all fee columns are added, all of them once the three bundled fees
are left out; Vendor 2 reconciles with all columns. The money fields are raw TLC values, and `total_amount − tip_amount`
is a consistent pre-tip bill for both vendors.

## 8. Time-based split

| Fold | Pickup dates | Days | Used for |
| --- | --- | --- | --- |
| train | Apr 1 – May 15, 2026 | 45 | fitting models, imputers, scalers, encoders |
| validation | May 16 – May 31, 2026 | 16 | tuning and model choice |
| test | June 1 – June 30, 2026 | 30 | final evaluation, touched once |

**Why by time and not at random.** Trips on the same day share weather, events, and traffic, so a random split would put
near-copies of test days into training and overstate performance. A time split also matches use: learn from the past,
evaluate on a later month. (The full leakage discussion is in the modeling part.)

In [ ]:
df["split"] = np.select([df["tpep_pickup_datetime"] < TRAIN_END, df["tpep_pickup_datetime"] < VAL_END],
                        ["train", "val"], default="test")
for c in V2_NEW:
    df[c] = v2[c]
df = df.sort_values("tpep_pickup_datetime").reset_index(drop=True)

split_summary = (df.groupby("split")
                   .agg(rows=("high_tip", "size"), days=("pickup_date", "nunique"),
                        first_day=("pickup_date", "min"), last_day=("pickup_date", "max"),
                        high_tip_rate=("high_tip", "mean"), high_tip_bill_rate=("high_tip_bill", "mean"))
                   .reindex(["train", "val", "test"]))
split_summary["share_of_rows"] = split_summary["rows"] / split_summary["rows"].sum()
display(split_summary.style.format({"rows": "{:,}", "high_tip_rate": "{:.1%}", "high_tip_bill_rate": "{:.1%}", "share_of_rows": "{:.1%}"}))

assert split_summary["rows"].to_dict() == EXPECTED_SPLIT_ROWS, split_summary["rows"].to_dict()
assert split_summary["high_tip_rate"].round(3).to_dict() == EXPECTED_SPLIT_HIGH_TIP, split_summary["high_tip_rate"].round(3).to_dict()
assert split_summary["days"].tolist() == [45, 16, 30]
print("Split counts, days, and high-tip rates match the fixed values.")

**Conclusion.** Train 395,144 / validation 127,349 / test 274,496 trips (49.6% / 16.0% / 34.4%). The high-tip rate is
63.3% / 62.4% / 63.0%, so the folds are comparable and no fold is starved (the reservoir bug would have left validation
with ~11k trips).

## 9. Output, data dictionary, and check against the fixed table

The rebuilt table has 796,989 rows × 54 columns. It is **not saved** here: the project's saved copy is
`MyDrive/CIS5450_project/data/clean/trips_clean_v2.parquet` (and `trips_clean_v1.parquet`, its first 48 columns), and the
last cell proves the rebuilt table is identical to it.

**Feature column:** ✅ usable as a feature · ⚠️ usable after a train-only fix · ❌ never a feature.

In [ ]:
DICTIONARY = [
    # column, meaning, source, feature
    ("VendorID", "meter vendor: 1 = Creative Mobile, 2 = Curb", "TLC", "✅ categorical"),
    ("tpep_pickup_datetime", "pickup time (NYC local)", "TLC", "❌ use derived columns"),
    ("tpep_dropoff_datetime", "drop-off time", "TLC", "❌ use duration_min"),
    ("passenger_count", "passengers; 0 set to missing (3,047 trips)", "TLC", "⚠️ impute on train"),
    ("trip_distance", "miles, 0-100", "TLC", "✅"),
    ("RatecodeID", "1 standard, 2 JFK, 3 Newark, 4 Nassau/Westchester, 5 negotiated", "TLC", "✅ categorical"),
    ("store_and_fwd_flag", "Y = trip stored in the vehicle before upload", "TLC", "✅ (little information)"),
    ("PULocationID", "pickup taxi zone 1-263", "TLC", "✅ categorical"),
    ("DOLocationID", "drop-off taxi zone 1-263", "TLC", "✅ categorical"),
    ("payment_type", "always 1 (card)", "TLC", "❌ constant"),
    ("fare_amount", "time-and-distance fare, > 0; denominator of tip_pct", "TLC", "✅"),
    ("extra", "rush $2.50 / overnight $1; Vendor 1 also bundles congestion, CBD, airport fees", "TLC", "⚠️ de-duplicate Vendor 1"),
    ("mta_tax", "MTA tax, almost always $0.50", "TLC", "✅"),
    ("tip_amount", "card tip, > 0 and <= fare", "TLC", "❌ outcome"),
    ("tolls_amount", "tolls", "TLC", "✅"),
    ("improvement_surcharge", "improvement surcharge, almost always $1", "TLC", "✅"),
    ("total_amount", "total charged, INCLUDES the tip", "TLC", "❌ contains outcome"),
    ("congestion_surcharge", "NY State congestion surcharge, $0 or $2.50", "TLC", "✅"),
    ("Airport_fee", "JFK / LGA pickup fee", "TLC", "✅"),
    ("cbd_congestion_fee", "Manhattan CBD congestion fee, $0 or $0.75", "TLC", "✅ (or has_cbd_fee)"),
    ("source_month", "month file the trip was sampled from", "derived", "❌ bookkeeping"),
    ("duration_min", "drop-off minus pickup, 1-180 min", "derived", "✅"),
    ("PU_Borough", "pickup borough", "zone join", "✅ categorical"),
    ("PU_Zone", "pickup zone name", "zone join", "⚠️ duplicates PULocationID"),
    ("PU_service_zone", "Yellow Zone / Airports / Boro Zone / EWR", "zone join", "✅ categorical"),
    ("DO_Borough", "drop-off borough", "zone join", "✅ categorical"),
    ("DO_Zone", "drop-off zone name", "zone join", "⚠️ duplicates DOLocationID"),
    ("DO_service_zone", "as PU_service_zone, drop-off", "zone join", "✅ categorical"),
    ("pickup_hour_ts", "pickup time floored to the hour (weather join key)", "derived", "❌ join key"),
    ("temp_c", "temperature in the pickup hour (°C)", "weather join", "✅"),
    ("precip_mm", "precipitation in the hour ending at pickup_hour_ts (mm)", "weather join", "✅"),
    ("is_rain", "precip_mm >= 0.1", "derived", "✅"),
    ("pickup_hour", "pickup hour 0-23", "derived", "✅"),
    ("pickup_dow", "day of week, 0 = Monday", "derived", "✅"),
    ("is_weekend", "Saturday or Sunday", "derived", "✅"),
    ("is_late_night", "pickup 00:00-05:59 (subset of is_night)", "derived", "✅"),
    ("avg_speed_mph", "60 x distance / duration", "derived", "⚠️ cap chosen on train"),
    ("is_airport", "pickup or drop-off at JFK (132) / LGA (138)", "derived", "✅"),
    ("is_newark", "pickup or drop-off at Newark (1)", "derived", "✅"),
    ("same_borough", "pickup borough == drop-off borough", "derived", "✅"),
    ("has_cbd_fee", "cbd_congestion_fee > 0", "derived", "✅"),
    ("is_short_trip", "trip_distance <= 2 mi", "derived", "✅"),
    ("tip_pct", "tip_amount / fare_amount, <= 1", "derived", "❌ outcome"),
    ("high_tip", "tip_pct >= 0.25 (proposal target)", "derived", "❌ target"),
    ("high_tip_20", "tip_pct >= 0.20 (sensitivity)", "derived", "❌ target"),
    ("high_tip_30", "tip_pct >= 0.30 (sensitivity)", "derived", "❌ target"),
    ("pickup_date", "pickup date (cluster unit for SEs / bootstrap)", "derived", "❌ bookkeeping"),
    ("split", "train / val / test by pickup date", "derived", "❌ bookkeeping"),
    ("is_rush_hour", "Mon-Fri pickup 16:00-19:59, holidays included", "v2", "✅"),
    ("is_night", "pickup 20:00-05:59", "v2", "✅"),
    ("tip_pct_bill", "tip / (total_amount - tip_amount)", "v2", "❌ outcome"),
    ("high_tip_bill", "tip > round(0.20 x bill, 2) + 0.015", "v2", "❌ target"),
    ("high_tip_bill_25", "same rule at 25%", "v2", "❌ target"),
    ("high_tip_bill_30", "same rule at 30%", "v2", "❌ target"),
]

dictionary = pd.DataFrame(DICTIONARY, columns=["column", "meaning", "source", "feature"]).set_index("column")
undocumented = [c for c in df.columns if c not in dictionary.index]
if undocumented:
    print("WARNING: columns without a dictionary entry:", undocumented)
dictionary = dictionary.reindex(df.columns).fillna("?")
dictionary.insert(0, "dtype", df.dtypes.astype(str))
dictionary.insert(1, "missing", df.isna().sum())
dictionary = dictionary.reset_index().rename(columns={"index": "column"})
dictionary.index = np.arange(1, len(dictionary) + 1)
display(dictionary)

NOT_FEATURES = dictionary.loc[dictionary["feature"].str.startswith("❌"), "column"].tolist()
print(f"{len(NOT_FEATURES)} columns must never be model features:", NOT_FEATURES)

### Is the rebuilt table identical to the fixed table the project uses?

Both tables are sorted the same way (by many columns, so the order is unique), then every column is compared value by
value. Weather columns are reported separately, because Open-Meteo can revise historical data; that would change
`temp_c` / `precip_mm` without changing which trips are in the table.

In [ ]:
KEY_COLS = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID", "VendorID",
            "trip_distance", "fare_amount", "tip_amount", "total_amount", "extra", "tolls_amount",
            "passenger_count", "RatecodeID", "mta_tax", "improvement_surcharge",
            "congestion_surcharge", "Airport_fee", "cbd_congestion_fee"]
WEATHER_COLS = ["temp_c", "precip_mm", "is_rain"]

def canonical(d):
    d = d.copy()
    for c in d.columns:
        if pd.api.types.is_datetime64_any_dtype(d[c]):
            d[c] = d[c].astype("datetime64[ns]")
    return d.sort_values(KEY_COLS, kind="mergesort", na_position="last").reset_index(drop=True)

if FIXED_V2.exists():
    fixed = pd.read_parquet(FIXED_V2)                      # read only
    print("rebuilt:", df.shape, "| fixed v2 on Drive:", fixed.shape)
    assert fixed.shape == df.shape, "different shape"
    assert list(fixed.columns) == list(df.columns), "different columns"
    A, B = canonical(df), canonical(fixed)
    identical = {}
    for c in df.columns:
        try:
            pd.testing.assert_series_equal(A[c], B[c], check_dtype=False, check_exact=True, check_names=False)
            identical[c] = True
        except AssertionError:
            identical[c] = False
    identical = pd.Series(identical)
    differ = identical[~identical].index.tolist()
    print("Columns that differ:", differ if differ else "none")
    assert identical.drop(WEATHER_COLS).all(), f"rebuilt table differs from the fixed table in {differ}"
    if not identical[WEATHER_COLS].all():
        print("NOTE: weather values differ from the saved table (Open-Meteo revised its data); the trips are identical.")
    print(f"IDENTICAL: the rebuilt table is the same {len(df):,} trips x {df.shape[1]} columns as {FIXED_V2.name}.")
else:
    print("Fixed table not found on Drive, so only the counts above could be checked.")
con.close()

### Summary

- **Data.** NYC TLC yellow taxi records for April–June 2026 (11,759,324 trips; 7,796,205 card trips), the Taxi Zone
  Lookup Table, and Open-Meteo hourly weather, downloaded from the official links in this notebook.
- **Sampling.** DuckDB, card trips only, 300,000 per month with `SEED = 5450` → 900,000 trips. Hash-based sampling replaced
  reservoir sampling after the reservoir sample proved time-biased (validation fold of 10,836 trips); every day's share in
  the sample is within about 0.06 pp of the full month.
- **Cleaning.** Eight rules, each with a reason and a count: 900,000 → 825,233 (positive tip and fare) → **796,989** trips
  (88.6%). Zero tips (74,765) and implausible durations (20,509) account for most removals. Slow trips are kept on purpose;
  `passenger_count = 0` becomes missing rather than dropping the trip.
- **Joins.** Zone (pickup + drop-off) and hourly weather LEFT JOINs both match 100% of trips with no row-count change;
  weather is aligned to the last completed hour before pickup. 16.2% of trips are in rainy hours.
- **Features.** Proposal Table 1 features, the `high_tip` target, and the v2 columns `is_rush_hour` (holidays included, as
  the meters charged the surcharge), `is_night`, and the bill-based outcome `tip_pct_bill` / `high_tip_bill` (+25%, 30%).
- **Money fields.** Raw TLC values, unchanged. Vendor 1's `extra` bundles three fees, so the bill is `total − tip` and
  `extra` must be de-duplicated before use as a feature.
- **Split.** By pickup date: train 395,144 / validation 127,349 / test 274,496; high-tip rate 63.3% / 62.4% / 63.0%.
- **Reproducibility.** Rebuilding from the official links gives a table identical to the fixed `trips_clean_v2.parquet`.
- **Leakage guard.** `tip_amount`, `total_amount`, and everything derived from them are never features; nothing in this
  part learns a statistic from the data, so imputation, capping, encoding, and scaling are fitted on train in the modeling part.

**Limitations.** Trips with a zero card tip are excluded, so results describe riders who tipped by card, not all riders.
91% of pickups are in Manhattan, so outer-borough patterns are imprecise. Weather is a single Manhattan point.
Early weekday mornings (Mon–Wed, ~2–5 AM) have only ~150–330 sampled trips per hour-by-day cell.